# Imports

In [1]:
import torch
from torchvision import transforms
from torchvision.models import resnet18, resnet50, resnet101
from torchvision.models import ResNet18_Weights, ResNet50_Weights, ResNet101_Weights
import requests
from PIL import Image
from io import BytesIO
import time

# Preprocess

In [2]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Labels

In [3]:
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(labels_url).text.split("\n")

# Models

In [4]:
models_dict = {
    "ResNet18": resnet18(weights=ResNet18_Weights.DEFAULT).eval(),
    "ResNet50": resnet50(weights=ResNet50_Weights.DEFAULT).eval(),
    "ResNet101": resnet101(weights=ResNet101_Weights.DEFAULT).eval()
}

# Uploading Images

In [5]:
def load_image(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Error loading image: {response.status_code}")

    return Image.open(BytesIO(response.content)).convert("RGB")

# Classification

In [6]:
def classify_image(model, url, top_k=5):
    img = load_image(url)
    img_tensor = preprocess(img).unsqueeze(0)

    start = time.time()

    with torch.no_grad():
        output = model(img_tensor)

    inference_time = time.time() - start

    probs = torch.nn.functional.softmax(output[0], dim=0)
    top_probs, top_idxs = torch.topk(probs, top_k)

    results = []
    for prob, idx in zip(top_probs, top_idxs):
        results.append({
            "class": labels[idx.item()],
            "prob": prob.item()
        })

    return results, inference_time

# Data

In [7]:
image_urls = [
    "https://images.unsplash.com/photo-1517849845537-4d257902454a",  # dog
    "https://images.unsplash.com/photo-1518791841217-8f162f1e1131",  # cat
    "https://images.unsplash.com/photo-1502877338535-766e1452684a",  # car
    "https://images.unsplash.com/photo-1517336714731-489689fd1ca8",  # laptop
    "https://images.unsplash.com/photo-1493238792000-8113da705763"   # airplane
]